# Weed Detection - GPU training on Google Colab

Trains **YOLO11** on the `Francesco/weed-crop-aerial` dataset using Colab's free GPU
(~50-100x faster than the CPU run). At the end it evaluates on the test split and
lets you download `best.pt` + the metrics/plots.

**Before you start:** `Runtime` menu -> `Change runtime type` -> Hardware accelerator = **T4 GPU** -> Save.

Then `Runtime` -> `Run all`. Takes roughly 30-60 min for 100 epochs on a T4.

In [ ]:
# 1. Check the GPU is on
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > T4 GPU, then Run all again.'
print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi -L

In [ ]:
# 2. Get the project code + install deps (Colab already has a matching torch)
!git clone --depth 1 https://github.com/Bhawna109/AI-based-Weed-detection.git repo
%cd repo
!pip -q install ultralytics datasets
import ultralytics; ultralytics.checks()

In [ ]:
# 3. Download + convert the dataset to YOLO format (writes configs/data.yaml)
#    On Colab the dataset goes in the working dir - no OneDrive, fast local disk.
!python src/fetch_hf_dataset.py --dataset Francesco/weed-crop-aerial
!python src/verify_dataset.py --samples 0

In [ ]:
# 4. Train.  GPU lets us use the bigger model, more epochs, bigger batch, RAM cache.
#    Change --model to yolo11n.pt / yolo11m.pt to trade speed for accuracy.
!python src/train.py \
    --model yolo11s.pt \
    --epochs 100 \
    --batch 32 \
    --imgsz 640 \
    --device 0 \
    --workers 2 \
    --cache ram \
    --patience 25 \
    --name weed_yolo11s_colab

In [ ]:
# 5. Evaluate on the untouched test split -> real Precision / Recall / mAP
!python src/evaluate.py \
    --weights results/runs/weed_yolo11s_colab/weights/best.pt \
    --split test --name test_eval_colab

In [ ]:
# 6. Run predictions on the test images and show a few
!python src/predict.py \
    --weights results/runs/weed_yolo11s_colab/weights/best.pt \
    --source dataset/images/test --conf 0.25 --name colab_test

import glob, random
from IPython.display import Image, display
imgs = glob.glob('results/predictions/colab_test/*.jpg')
for p in random.sample(imgs, min(6, len(imgs))):
    display(Image(filename=p, width=500))

In [ ]:
# 7. Show the training curves + confusion matrix
!python src/plot_results.py --run results/runs/weed_yolo11s_colab
from IPython.display import Image, display
for p in ['results/training_curves.png',
          'results/test_confusion_matrix_normalized.png',
          'results/test_PR_curve.png']:
    try: display(Image(filename=p, width=650))
    except Exception as e: print(p, e)

In [ ]:
# 8. Package the weights + metrics + plots and download to your computer
import shutil, os
os.makedirs('download', exist_ok=True)
shutil.copy('results/runs/weed_yolo11s_colab/weights/best.pt', 'download/best.pt')
for f in ['metrics_test.json', 'training_curves.png',
          'test_PR_curve.png', 'test_confusion_matrix_normalized.png']:
    src = f'results/{f}'
    if os.path.exists(src): shutil.copy(src, f'download/{f}')
shutil.make_archive('weed_yolo11s_colab', 'zip', 'download')
print('size:', round(os.path.getsize('weed_yolo11s_colab.zip')/1e6, 1), 'MB')
from google.colab import files
files.download('weed_yolo11s_colab.zip')

## After download

On your PC, unzip and drop the files in:

- `best.pt` -> `results/runs/weed_yolo11s_colab/weights/best.pt` (make the folders)
- the `.png` / `.json` -> `results/`

then update the Results table in `README.md` with the new numbers and commit.
Run local predictions any time with:

```bash
python src/predict.py --weights results/runs/weed_yolo11s_colab/weights/best.pt --source <image_or_folder>
```